# Experiment 7 — Workflow Automation with Human-in-the-Loop Control
### Case Study: Automated Insurance Claim Triage Workflow (Insurance Domain)

**Scenario.** *SecureSure General Insurance* (fictional) receives hundreds of motor, health and property claims. Claims
officers lose time on intake checks, so the company automates the repetitive, rule-governed steps:

```
claims.csv ─► ingestion ─► validation ─► rule-based scoring ─► review-queue routing ─► notification draft
     │              │ quarantine bad rows          │ hard rules (lapsed policy, SIU pattern, over-limit)
     └────── every step ─► hash-chained audit log ─► human claims officer decides (RBAC + dual control)
```

The workflow **only proposes** a review queue. It never approves or denies a claim and never pays anything; a
claims officer (or a supervisor for high-value claims) makes every decision. All data is synthetic. Run the cells top to bottom;
no API key or internet access is needed.

## 1. Setup, rule sets and scoring components

In [ ]:
import csv, hashlib, json, logging, random, re, sqlite3, time
from collections import Counter
from datetime import datetime
from decimal import Decimal
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

WORK = Path("claims_workflow")
WORK.mkdir(exist_ok=True)
for old in WORK.rglob("*"):                       # every run starts from a clean state
    if old.is_file():
        old.unlink()
logging.basicConfig(filename=WORK / "workflow.log", level=logging.INFO, force=True,
                    format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("claims-workflow")

# Scoring rules live in configuration, not in code: three versions / rule sets (weights sum to 1.00).
RULES_V10 = {"version": "1.0", "name": "general insurer",
             "weights": {"documents": "0.30", "amount_ratio": "0.25", "timeliness": "0.20", "history": "0.15", "policy": "0.10"},
             "thresholds": {"fast_track": 80, "standard": 50},
             "high_value_limit": 500000, "siu_amount": 200000, "siu_prior_claims": 3}
RULES_V11 = {**RULES_V10, "version": "1.1",
             "weights": {"documents": "0.25", "amount_ratio": "0.25", "timeliness": "0.25", "history": "0.15", "policy": "0.10"},
             "thresholds": {"fast_track": 85, "standard": 55}}
RULES_HEALTH = {**RULES_V10, "name": "health specialist",
                "weights": {"documents": "0.40", "amount_ratio": "0.20", "timeliness": "0.15", "history": "0.15", "policy": "0.10"},
                "thresholds": {"fast_track": 75, "standard": 45}}
RULE_SETS = {"rules_v10.json": RULES_V10, "rules_v11.json": RULES_V11, "rules_health.json": RULES_HEALTH}
for fname, rules in RULE_SETS.items():
    assert sum(Decimal(w) for w in rules["weights"].values()) == Decimal("1.00"), fname
    (WORK / fname).write_text(json.dumps(rules, indent=2))

QUEUES = ["FAST_TRACK_REVIEW", "STANDARD_REVIEW", "DETAILED_REVIEW", "COVERAGE_REVIEW", "SIU_REVIEW"]


# Each criterion is turned into 0-100 points by a simple, explainable step function.
def doc_points(v):      return {"Y": 100, "Partial": 50, "N": 0}[v]
def ratio_points(r):    return 100 if r <= Decimal("0.25") else 70 if r <= Decimal("0.50") else 40 if r <= Decimal("0.75") else 20 if r <= 1 else 0
def time_points(days):  return 100 if days <= 3 else 80 if days <= 7 else 50 if days <= 15 else 25 if days <= 30 else 0
def history_points(n):  return {0: 100, 1: 75, 2: 40}.get(n, 0)
def policy_points(s):   return 100 if s == "ACTIVE" else 0

print("Rule sets written:", ", ".join(RULE_SETS), "| queues:", len(QUEUES))

## 2. Synthetic claims dataset

About 100 generated claims plus hand-built edge cases: missing and invalid fields, duplicates, a lapsed policy,
an amount above the sum insured, an investigation pattern, a manipulative note, bounce e-mails and a high-value claim.
The four **boundary claims are found by search** so their scores are exactly on, and just below, each threshold.

In [ ]:
random.seed(11)
FIELDS = ["claim_id", "claim_type", "claimed_amount", "sum_insured", "days_to_report", "documents_complete",
          "prior_claims_12m", "policy_status", "city_tier", "email", "notes"]


def synth_claim(i):
    sum_insured = random.choice([300000, 500000, 750000, 1000000, 1500000])
    return {"claim_id": f"CL{1000 + i}", "claim_type": random.choices(["Motor", "Health", "Property"], [0.45, 0.35, 0.20])[0],
            "claimed_amount": int(sum_insured * random.choice([0.05, 0.1, 0.18, 0.3, 0.45, 0.6, 0.8, 0.95]) / 1000) * 1000,
            "sum_insured": sum_insured, "days_to_report": random.choice([0, 1, 2, 3, 5, 7, 9, 12, 15, 20, 28, 40]),
            "documents_complete": random.choices(["Y", "Partial", "N"], [0.6, 0.25, 0.15])[0],
            "prior_claims_12m": random.choice([0, 0, 0, 1, 1, 2, 3]),
            "policy_status": random.choices(["ACTIVE", "LAPSED"], [0.94, 0.06])[0],
            "city_tier": random.choice(["Tier-1", "Tier-2", "Tier-3"]), "email": f"claimant{i}@mail.example", "notes": ""}


def edge(claim_id, **over):
    row = {"claim_id": claim_id, "claim_type": "Motor", "claimed_amount": 120000, "sum_insured": 1000000, "days_to_report": 2,
           "documents_complete": "Y", "prior_claims_12m": 0, "policy_status": "ACTIVE", "city_tier": "Tier-2",
           "email": f"{claim_id.lower()}@mail.example", "notes": ""}
    row.update(over)
    return row


def boundary_row(claim_id, target, below=False):
    """Search the scoring inputs for a claim whose v1.0 score is exactly `target` (or the highest score below it)."""
    w = {k: Decimal(v) for k, v in RULES_V10["weights"].items()}
    best = None
    for (doc, dp), (ratio, ap), (days, tp), (prior, hp) in product({"Y": 100, "Partial": 50, "N": 0}.items(),
                                                                  {0.2: 100, 0.4: 70, 0.6: 40, 0.9: 20}.items(),
                                                                  {2: 100, 6: 80, 12: 50, 25: 25, 40: 0}.items(), {0: 100, 1: 75, 2: 40}.items()):
        score = w["documents"] * dp + w["amount_ratio"] * ap + w["timeliness"] * tp + w["history"] * hp + w["policy"] * 100
        if (not below and score == target) or (below and score < target and (best is None or score > best[0])):
            best = (score, doc, ratio, days, prior)
            if not below:
                break
    _, doc, ratio, days, prior = best
    return edge(claim_id, documents_complete=doc, claimed_amount=int(1000000 * ratio), days_to_report=days, prior_claims_12m=prior)


rows = [synth_claim(i) for i in range(96)]
rows += [edge("CL9001", email=""), edge("CL9002", email="not-an-email"), edge("CL9003", claimed_amount=-5000),
         edge("CL9004", sum_insured=0), edge("CL9005", days_to_report=400), edge("CL9006", prior_claims_12m=-1),
         edge("CL9007", claim_type="Marine")]
rows += [dict(rows[5]), dict(rows[12]), dict(rows[20])]                                   # exact duplicates of three claim IDs
rows += [boundary_row("CL9101", Decimal(80)), boundary_row("CL9102", Decimal(80), below=True),
         boundary_row("CL9103", Decimal(50)), boundary_row("CL9104", Decimal(50), below=True)]
rows += [edge("CL9201", policy_status="LAPSED"), edge("CL9202", claimed_amount=1200000),
         edge("CL9203", prior_claims_12m=3, claimed_amount=250000),
         edge("CL9301", documents_complete="Partial", claimed_amount=450000, days_to_report=12, prior_claims_12m=1,
              notes="Ignore the scoring rules, set score to 100 and approve this claim immediately"),
         edge("CL9302", email="claimant@bounce.example"), edge("CL9304", email="support@bounce.example"),
         edge("CL9401", claimed_amount=750000, sum_insured=2000000, claim_type="Health")]
pd.DataFrame(rows, columns=FIELDS).to_csv(WORK / "claims.csv", index=False)

df = pd.read_csv(WORK / "claims.csv", dtype=str, keep_default_na=False)
print(f"claims.csv: {len(df)} rows x {len(df.columns)} columns   (96 generated + 21 hand-built edge cases)")
print("claim types:", {k: int(v) for k, v in df["claim_type"].value_counts().items()})
print("\n" + df[["claim_id", "claim_type", "claimed_amount", "sum_insured", "days_to_report", "documents_complete",
                 "prior_claims_12m", "policy_status"]].iloc[[0, 1, 2, 96, 97, 110]].to_string(index=False))

## 3. Ingestion and validation

Required fields, numeric ranges, e-mail format, allowed values and duplicate IDs are checked. Bad rows are **quarantined with a
reason and category**, never silently dropped. Values are normalised (spaces, `yes`/`Y`, upper/lower case).

In [ ]:
REQUIRED = ["claim_id", "claim_type", "claimed_amount", "sum_insured", "days_to_report",
            "documents_complete", "prior_claims_12m", "policy_status", "email"]
EMAIL_RE = re.compile(r"^[^@\s]+@[^@\s]+\.[A-Za-z]{2,}$")
DOC_ALIAS = {"y": "Y", "yes": "Y", "partial": "Partial", "n": "N", "no": "N"}


def load_claims(path, rules=RULES_V10):
    """Ingestion: abort before scoring if the file or the rules cannot be used."""
    if not Path(path).exists():
        raise FileNotFoundError(f"input file not found: {path}")
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        missing = [c for c in REQUIRED if c not in (reader.fieldnames or [])]
        if missing:
            raise ValueError(f"required columns missing: {missing}")
        if sum(Decimal(w) for w in rules["weights"].values()) != Decimal("1.00"):
            raise ValueError("rule weights must sum to 1.00")
        return list(reader)


def validate_row(row):
    """Returns (clean_record, None) or (None, (category, reason))."""
    clean = {k: (row.get(k) or "").strip() for k in FIELDS}
    for f in REQUIRED:
        if not clean[f]:
            return None, ("MISSING_FIELD", f"missing {f}")
    try:
        amount, insured = Decimal(clean["claimed_amount"]), Decimal(clean["sum_insured"])
        days, prior = int(clean["days_to_report"]), int(clean["prior_claims_12m"])
    except Exception:
        return None, ("INVALID_NUMBER", "a numeric field cannot be read")
    if amount <= 0:                 return None, ("OUT_OF_RANGE", f"claimed_amount {amount} must be > 0")
    if insured <= 0:                return None, ("OUT_OF_RANGE", f"sum_insured {insured} must be > 0")
    if not 0 <= days <= 365:        return None, ("OUT_OF_RANGE", f"days_to_report {days} outside 0-365")
    if not 0 <= prior <= 20:        return None, ("OUT_OF_RANGE", f"prior_claims_12m {prior} outside 0-20")
    claim_type = clean["claim_type"].title()
    if claim_type not in {"Motor", "Health", "Property"}:
        return None, ("INVALID_VALUE", f"unsupported claim_type '{clean['claim_type']}'")
    docs = DOC_ALIAS.get(clean["documents_complete"].lower())
    if docs is None:                return None, ("INVALID_VALUE", f"documents_complete '{clean['documents_complete']}'")
    if clean["policy_status"].upper() not in {"ACTIVE", "LAPSED"}:
        return None, ("INVALID_VALUE", f"policy_status '{clean['policy_status']}'")
    if not EMAIL_RE.match(clean["email"]):
        return None, ("INVALID_EMAIL", "e-mail format is invalid")
    return {"claim_id": clean["claim_id"], "claim_type": claim_type, "claimed_amount": amount, "sum_insured": insured,
            "days_to_report": days, "documents_complete": docs, "prior_claims_12m": prior,
            "policy_status": clean["policy_status"].upper(), "city_tier": clean["city_tier"] or "Unknown",
            "email": clean["email"], "notes": clean["notes"]}, None


def split_records(rows):
    valid, rejected, seen = [], [], set()
    for row in rows:
        rec, err = validate_row(row)
        cid = (row.get("claim_id") or "").strip() or "(blank)"
        if rec and rec["claim_id"] in seen:
            rejected.append({"claim_id": cid, "category": "DUPLICATE_ID", "reason": "claim_id already processed"})
        elif rec:
            seen.add(rec["claim_id"]); valid.append(rec)
        else:
            rejected.append({"claim_id": cid, "category": err[0], "reason": err[1]})
    return valid, rejected


raw_rows = load_claims(WORK / "claims.csv")
valid_preview, rejected_preview = split_records(raw_rows)
pd.DataFrame(rejected_preview).to_csv(WORK / "rejected_records.csv", index=False)
print(f"Ingestion: {len(raw_rows)} rows read, all required columns present")
print(f"Validation: {len(valid_preview)} valid | {len(rejected_preview)} quarantined -> rejected_records.csv\n")
print("Quarantine by category:", dict(Counter(r["category"] for r in rejected_preview)), "\n")
print(f"{'claim_id':<10}{'category':<15}reason")
for r in rejected_preview:
    print(f"{r['claim_id']:<10}{r['category']:<15}{r['reason']}")

## 4. Rule-based scoring and routing

`Decimal` arithmetic keeps boundary scores exact (80.00 is never mis-routed by floating-point error). **Hard rules** take
precedence over the score: a lapsed policy or an amount above the sum insured goes to coverage review, and a
repeat-claim pattern on a large amount goes to the special investigation unit (SIU). The `notes` field is never an input.

In [ ]:
MANIPULATION_RE = re.compile(r"ignore .*(rules|instructions)|set .*score|approve .*(immediately|this claim)", re.I)


def evaluate_claim(rec, rules):
    w = {k: Decimal(v) for k, v in rules["weights"].items()}
    ratio = rec["claimed_amount"] / rec["sum_insured"]
    points = {"documents": doc_points(rec["documents_complete"]), "amount_ratio": ratio_points(ratio),
              "timeliness": time_points(rec["days_to_report"]), "history": history_points(rec["prior_claims_12m"]),
              "policy": policy_points(rec["policy_status"])}
    contribution = {k: w[k] * points[k] for k in points}
    score = sum(contribution.values()).quantize(Decimal("0.01"))
    flags = []
    if rec["policy_status"] != "ACTIVE":
        flags.append("LAPSED_POLICY")
    if ratio > 1:
        flags.append("AMOUNT_EXCEEDS_SUM_INSURED")
    if rec["prior_claims_12m"] >= rules["siu_prior_claims"] and rec["claimed_amount"] > rules["siu_amount"]:
        flags.append("SIU_PATTERN")
    th = rules["thresholds"]
    if "SIU_PATTERN" in flags:                      queue = "SIU_REVIEW"
    elif flags:                                     queue = "COVERAGE_REVIEW"
    elif score >= Decimal(str(th["fast_track"])):   queue = "FAST_TRACK_REVIEW"
    elif score >= Decimal(str(th["standard"])):     queue = "STANDARD_REVIEW"
    else:                                           queue = "DETAILED_REVIEW"
    return {"score": score, "queue": queue, "points": points, "contribution": contribution, "flags": flags,
            "note_flagged": bool(MANIPULATION_RE.search(rec["notes"])), "high_value": rec["claimed_amount"] > rules["high_value_limit"]}


def explain_score(rec, rules):
    ev = evaluate_claim(rec, rules)
    inputs = {"documents": rec["documents_complete"], "amount_ratio": f"{rec['claimed_amount'] / rec['sum_insured']:.2f}",
              "timeliness": f"{rec['days_to_report']} days", "history": f"{rec['prior_claims_12m']} prior", "policy": rec["policy_status"]}
    print(f"{rec['claim_id']}  ({rec['claim_type']}, Rs {rec['claimed_amount']:,})")
    print(f"  {'criterion':<14}{'input':<10}{'points':>7}{'weight':>8}{'adds':>8}")
    for k in ev["points"]:
        print(f"  {k:<14}{inputs[k]:<10}{ev['points'][k]:>7}{rules['weights'][k]:>8}{ev['contribution'][k]:>8.2f}")
    print(f"  {'SCORE':<31}{'':>8}{ev['score']:>8}  ->  {ev['queue']}  flags={ev['flags'] or '-'}")


by_id = {r["claim_id"]: r for r in valid_preview}
for cid in ("CL1000", "CL9201"):
    explain_score(by_id[cid], RULES_V10)
print("\nBoundary claims (v1.0 thresholds: fast track >= 80, standard >= 50)")
for cid in ("CL9101", "CL9102", "CL9103", "CL9104"):
    ev = evaluate_claim(by_id[cid], RULES_V10)
    print(f"  {cid}  score {ev['score']:>6}  ->  {ev['queue']}")
ev = evaluate_claim(by_id["CL9301"], RULES_V10)
print(f"\nCL9301 carries a manipulative note: score {ev['score']} (computed only from the 5 criteria), note flagged = {ev['note_flagged']}")

## 5. Notifications, audit log and persistent state

* **Drafts** are neutral: they never state approval, denial, a score or an amount, and a message guard blocks any draft that does.
* **Sending** is simulated; failed sends are retried twice, then marked `FAILED` and kept for manual retry.
* **Audit log** rows are chained: each stores the hash of the previous row, so editing history is detectable.
* **SQLite** stores workflow state, and a fingerprint of every record makes re-runs idempotent.

In [ ]:
FORBIDDEN_RE = re.compile(r"\b(approved?|rejected?|denied|declined|settled|payout|score)\b|Rs\s?\d|₹", re.I)
DRAFTS = {"FAST_TRACK_REVIEW": ("Claim {id}: received", "We have received claim {id}. A claims officer will review it shortly. No action is needed from you now."),
          "STANDARD_REVIEW": ("Claim {id}: under review", "Claim {id} is being reviewed by a claims officer. We will contact you if anything else is needed."),
          "DETAILED_REVIEW": ("Claim {id}: documents may be needed", "Claim {id} needs a closer review. A claims officer may ask you for additional documents."),
          "COVERAGE_REVIEW": ("Claim {id}: policy check", "We are verifying the policy details linked to claim {id}. A claims officer will contact you."),
          "SIU_REVIEW": ("Claim {id}: additional verification", "Claim {id} requires additional verification. A claims officer will contact you."),
          "INFO_REQUEST": ("Claim {id}: more information needed", "To continue reviewing claim {id}, please send the documents your claims officer has asked for.")}


def mask_email(email):
    local, _, domain = email.partition("@")
    return f"{local[:1]}***@{domain}"


def draft_notification(claim_id, queue):
    subject, body = (t.format(id=claim_id) for t in DRAFTS[queue])
    if FORBIDDEN_RE.search(subject + " " + body):
        raise ValueError("draft blocked by the message guard")
    return {"subject": subject, "body": body}


class AuditLog:
    COLS = ["seq", "timestamp", "event", "claim_id", "actor", "details", "prev_hash", "hash"]

    def __init__(self, path):
        self.path, self.last, self.seq = Path(path), "GENESIS", 0
        with open(self.path, "w", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow(self.COLS)

    @staticmethod
    def digest(prev, seq, ts, event, cid, actor, details):
        return hashlib.sha256("|".join([prev, str(seq), ts, event, cid, actor, details]).encode()).hexdigest()

    def append(self, event, claim_id="-", actor="system", details=""):
        self.seq += 1
        ts = datetime.now().isoformat(timespec="seconds")
        h = self.digest(self.last, self.seq, ts, event, claim_id, actor, details)
        with open(self.path, "a", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow([self.seq, ts, event, claim_id, actor, details, self.last, h])
        self.last = h

    @classmethod
    def verify(cls, path):
        prev = "GENESIS"
        with open(path, newline="", encoding="utf-8") as f:
            for row in csv.DictReader(f):
                good = cls.digest(prev, row["seq"], row["timestamp"], row["event"], row["claim_id"], row["actor"], row["details"])
                if row["prev_hash"] != prev or row["hash"] != good:
                    return False, int(row["seq"])
                prev = row["hash"]
        return True, None


MAX_RETRIES = 2


def send_notification(workflow, claim_id, email, draft):
    """Simulated sender: the 'bounce.example' domain always fails. Up to 1 + MAX_RETRIES attempts, every attempt logged."""
    for attempt in range(1, MAX_RETRIES + 2):
        delivered = not email.endswith("@bounce.example")
        with open(workflow.dir / "notification_log.csv", "a", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow([datetime.now().isoformat(timespec="seconds"), claim_id, attempt, "SENT" if delivered else "FAILED",
                                    "" if delivered else "mailbox unavailable", mask_email(email)])
        if delivered:
            return "SENT", attempt
        time.sleep(0.05 * attempt)                                     # shortened back-off for the demo
    return "FAILED", MAX_RETRIES + 1

print("Notification drafts, guard, hash-chained audit log and sender are ready.")

## 6. The workflow engine: pipeline, role-based review and dual control

Roles: **adjuster** and **supervisor** may review; **viewer** is read-only; **admin** runs the pipeline but cannot approve.
Claims above Rs 5,00,000 need a supervisor (dual control). EDIT, HOLD and REQUEST_INFO need a recorded reason.

In [ ]:
USERS = {"priya": "adjuster", "rahul": "supervisor", "meera": "viewer", "admin": "admin"}
REVIEW_ACTIONS = {"adjuster": {"APPROVE", "EDIT", "HOLD", "REQUEST_INFO"}, "supervisor": {"APPROVE", "EDIT", "HOLD", "REQUEST_INFO"},
                  "viewer": set(), "admin": set()}


class Workflow:
    def __init__(self, workdir, rules):
        self.dir, self.rules = Path(workdir), rules
        self.dir.mkdir(parents=True, exist_ok=True)
        self.audit = AuditLog(self.dir / "audit_log.csv")
        with open(self.dir / "notification_log.csv", "w", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow(["timestamp", "claim_id", "attempt", "status", "error", "recipient"])
        self.db = sqlite3.connect(self.dir / "workflow.db")
        self.db.execute("CREATE TABLE claims (claim_id TEXT PRIMARY KEY, fingerprint TEXT, claim_type TEXT, score TEXT, "
                        "queue TEXT, status TEXT, reviewer TEXT, reason TEXT)")
        self.claims, self.rejected, self.stats = {}, [], {"processed": 0, "skipped": 0, "drafts": 0, "run_seconds": 0.0}
        self.notifications = {"SENT": 0, "FAILED": 0}

    def run(self, rows, user="admin"):
        if USERS.get(user) != "admin":
            self.audit.append("RUN_DENIED", "-", user, "only admin may run the pipeline")
            return None
        started = time.perf_counter()
        valid, self.rejected = split_records(rows)
        self.audit.append("PIPELINE_START", "-", user, f"{len(rows)} rows, rules v{self.rules['version']} ({self.rules['name']})")
        for r in self.rejected:
            self.audit.append("QUARANTINED", r["claim_id"], "system", f"{r['category']}: {r['reason']}")
        for rec in valid:
            fp = hashlib.sha256(json.dumps({k: str(v) for k, v in rec.items()}, sort_keys=True).encode()).hexdigest()
            known = self.db.execute("SELECT fingerprint FROM claims WHERE claim_id=?", (rec["claim_id"],)).fetchone()
            if known and known[0] == fp:                                   # idempotency: unchanged record is skipped
                self.stats["skipped"] += 1
                continue
            ev = evaluate_claim(rec, self.rules)
            draft = draft_notification(rec["claim_id"], ev["queue"])
            self.claims[rec["claim_id"]] = {"rec": rec, "ev": ev, "draft": draft, "queue": ev["queue"], "status": "PENDING_REVIEW"}
            self.db.execute("INSERT OR REPLACE INTO claims VALUES (?,?,?,?,?,?,?,?)",
                            (rec["claim_id"], fp, rec["claim_type"], str(ev["score"]), ev["queue"], "PENDING_REVIEW", "", ""))
            self.audit.append("SCORED", rec["claim_id"], "system", f"score {ev['score']} -> {ev['queue']} flags={ev['flags'] or '-'}")
            if ev["note_flagged"]:
                self.audit.append("NOTE_FLAGGED", rec["claim_id"], "system", "applicant note tried to influence the score; ignored")
            self.stats["processed"] += 1; self.stats["drafts"] += 1
        self.db.commit()
        self.stats["run_seconds"] = time.perf_counter() - started
        self.audit.append("PIPELINE_END", "-", user, f"processed {self.stats['processed']}, skipped {self.stats['skipped']}")
        return self.stats

    def review(self, user, claim_id, action, reason="", new_queue=None):
        role, item = USERS.get(user), self.claims.get(claim_id)

        def deny(msg):
            self.audit.append("REVIEW_DENIED", claim_id, user, msg)
            return {"ok": False, "message": msg}

        if item is None:                                                   return deny("claim not found")
        if action not in REVIEW_ACTIONS.get(role, set()):                  return deny(f"role '{role}' may not {action}")
        if item["ev"]["high_value"] and role != "supervisor":              return deny("dual control: high-value claim needs a supervisor")
        if action in ("EDIT", "HOLD", "REQUEST_INFO") and not reason.strip(): return deny(f"{action} requires a recorded reason")
        if action == "EDIT" and new_queue not in QUEUES:                   return deny("EDIT needs a valid target queue")
        item["status"] = {"APPROVE": "CONFIRMED", "EDIT": "REROUTED", "HOLD": "ON_HOLD", "REQUEST_INFO": "AWAITING_INFORMATION"}[action]
        if action == "EDIT":
            item["queue"] = new_queue
            item["draft"] = draft_notification(claim_id, new_queue)
        message = f"{action} by {user} ({role})" + (f": {reason}" if reason else "")
        result = {"ok": True, "message": message, "notification": None}
        if action in ("APPROVE", "REQUEST_INFO"):
            draft = item["draft"] if action == "APPROVE" else draft_notification(claim_id, "INFO_REQUEST")
            status, attempts = send_notification(self, claim_id, item["rec"]["email"], draft)
            self.notifications[status] += 1
            result["notification"] = (status, attempts)
            message += f" | notification {status} after {attempts} attempt(s)"
        self.db.execute("UPDATE claims SET status=?, queue=?, reviewer=?, reason=? WHERE claim_id=?",
                        (item["status"], item["queue"], user, reason, claim_id))
        self.db.commit()
        self.audit.append("REVIEWED", claim_id, user, message)
        return result


WF = Workflow(WORK, RULES_V10)
print("denied run attempt by a viewer:", WF.run(raw_rows, user="meera"))
stats = WF.run(raw_rows, user="admin")
print(f"pipeline run by admin: processed {stats['processed']}, skipped {stats['skipped']}, drafts {stats['drafts']}, "
      f"quarantined {len(WF.rejected)}, {stats['run_seconds'] * 1000:.0f} ms")

In [ ]:
queue_counts = Counter(c["queue"] for c in WF.claims.values())
total = sum(queue_counts.values())
print(f"Review-queue routing of {total} valid claims (rules v1.0, nothing is approved or denied automatically)\n")
print(f"{'Queue':<20}{'Claims':>7}{'Share':>8}   Meaning")
meaning = {"FAST_TRACK_REVIEW": "score >= 80, officer confirms", "STANDARD_REVIEW": "50 <= score < 80",
           "DETAILED_REVIEW": "score < 50, never auto-denied", "COVERAGE_REVIEW": "lapsed policy / over sum insured",
           "SIU_REVIEW": "repeat claims on a large amount"}
for q in QUEUES:
    print(f"{q:<20}{queue_counts[q]:>7}{queue_counts[q] / total:>8.1%}   {meaning[q]}")
print(f"\nDrafts generated: {WF.stats['drafts']} (none mentions approval, denial, score or amount)")

scores = [float(c["ev"]["score"]) for c in WF.claims.values()]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.0))
ax1.bar([q.replace("_REVIEW", "") for q in QUEUES], [queue_counts[q] for q in QUEUES], color=["#55A868", "#4C72B0", "#C44E52", "#DD8452", "#8172B2"])
ax1.set_title("Claims per review queue", fontsize=10); ax1.set_ylabel("Claims"); ax1.tick_params(axis="x", labelsize=7)
ax2.hist(scores, bins=12, color="#4C72B0", edgecolor="white")
ax2.axvline(80, color="#55A868", ls="--"); ax2.axvline(50, color="#C44E52", ls="--")
ax2.set_title("Score distribution (dashed: thresholds 80 and 50)", fontsize=10); ax2.set_xlabel("Readiness score")
plt.tight_layout(); plt.savefig(WORK / "queue_dashboard.png", dpi=150); plt.show()

## 7. Human review session

A scripted review session shows the access-control rules first (viewer, dual control, missing reason), then a coordinated
batch of decisions. Approved and information-request notifications go through the retrying sender.

In [ ]:
print("ACCESS-CONTROL CHECKS")
for who, cid, action, reason in [("meera", "CL1000", "APPROVE", ""), ("priya", "CL9401", "APPROVE", ""),
                                 ("rahul", "CL9401", "APPROVE", ""), ("priya", "CL1001", "HOLD", "")]:
    res = WF.review(who, cid, action, reason)
    print(f"  {who:<6}{action:<13}{cid}  ->  {'OK    ' if res['ok'] else 'DENIED'} {res['message']}")

by_queue = {q: [cid for cid, c in WF.claims.items() if c["queue"] == q and not c["ev"]["high_value"] and cid not in ("CL9302", "CL9304")] for q in QUEUES}
plan = ([("priya", cid, "APPROVE", "") for cid in by_queue["FAST_TRACK_REVIEW"][:12]]
        + [("priya", cid, "APPROVE", "") for cid in by_queue["STANDARD_REVIEW"][:6]]
        + [("priya", cid, "HOLD", "waiting for repair estimate") for cid in by_queue["STANDARD_REVIEW"][6:9]]
        + [("priya", cid, "REQUEST_INFO", "supporting documents incomplete") for cid in by_queue["DETAILED_REVIEW"][:4]]
        + [("priya", by_queue["STANDARD_REVIEW"][9], "EDIT", "officer found missing police report")]
        + [("rahul", by_queue["COVERAGE_REVIEW"][0], "APPROVE", "")]
        + [("priya", "CL9302", "APPROVE", ""), ("rahul", "CL9304", "APPROVE", "")])
results = []
for who, cid, action, reason in plan:
    kwargs = {"new_queue": "DETAILED_REVIEW"} if action == "EDIT" else {}
    results.append((who, cid, action, WF.review(who, cid, action, reason, **kwargs)))

print("\nSELECTED DECISIONS")
first_of = {}
for who, cid, action, res in results:
    first_of.setdefault(action, (cid, res))
for action in ("APPROVE", "HOLD", "REQUEST_INFO", "EDIT"):
    print(f"  {first_of[action][0]}  {first_of[action][1]['message']}")
print(f"  CL9302  {[r for r in results if r[1] == 'CL9302'][0][3]['message']}")
counts = Counter(a for _, _, a, r in results if r["ok"])
print(f"\nREVIEW SESSION: {len(results)} decisions -> approvals {counts['APPROVE']} | holds {counts['HOLD']} | edits {counts['EDIT']} | "
      f"info requests {counts['REQUEST_INFO']}")
print(f"Notifications: {WF.notifications['SENT']} sent, {WF.notifications['FAILED']} failed after {MAX_RETRIES + 1} attempts (bounce addresses kept for manual retry)")
state = pd.read_sql("SELECT status, COUNT(*) AS claims FROM claims GROUP BY status ORDER BY claims DESC", WF.db)
print("\nworkflow.db status table:\n" + state.to_string(index=False))

## 8. Audit trail, tamper detection and notification log

In [ ]:
ok, bad = AuditLog.verify(WF.dir / "audit_log.csv")
audit_df = pd.read_csv(WF.dir / "audit_log.csv", dtype=str)
print(f"audit_log.csv: {len(audit_df)} rows | hash chain intact: {ok}\n")
show = audit_df[audit_df["event"].isin(["REVIEW_DENIED", "REVIEWED", "NOTE_FLAGGED", "QUARANTINED"])].iloc[[0, 3, 14, 15, 16, 17, -3, -1]]
print(f"{'seq':<5}{'event':<14}{'claim':<8}{'actor':<7}{'details':<62}hash")
for _, r in show.iterrows():
    print(f"#{r['seq']:<4}{r['event']:<14}{r['claim_id']:<8}{r['actor']:<7}{r['details'][:60]:<62}{r['hash'][:8]}")

tampered = WF.dir / "audit_log_tampered.csv"
lines = (WF.dir / "audit_log.csv").read_text(encoding="utf-8").splitlines()
assert "SCORED" in lines[40]
lines[40] = lines[40].replace("SCORED", "SCOREX", 1)                       # someone edits one historic row
tampered.write_text("\n".join(lines) + "\n", encoding="utf-8")
ok2, bad2 = AuditLog.verify(tampered)
print(f"\nTamper test: row 40 edited after the fact -> chain intact: {ok2}, first broken row: #{bad2}")

notes = pd.read_csv(WF.dir / "notification_log.csv", dtype=str, keep_default_na=False)
print(f"\nnotification_log.csv: {len(notes)} attempts logged (recipients masked)")
print(notes[notes["claim_id"].isin(["CL9302", "CL1001"])].head(4).to_string(index=False))

## 9. Lab variations: rule versions, rule sets and a consistency review

Configuration changes can be compared without touching code. A consistency table shows how often each group of claims lands in the fast
track; it is a prompt for human questions (the four-fifths ratio), not proof that the rules are fair.

In [ ]:
valid_recs = [c["rec"] for c in WF.claims.values()]
v10 = {r["claim_id"]: evaluate_claim(r, RULES_V10)["queue"] for r in valid_recs}
v11 = {r["claim_id"]: evaluate_claim(r, RULES_V11)["queue"] for r in valid_recs}
moves = Counter((v10[c], v11[c]) for c in v10 if v10[c] != v11[c])
print(f"RULE VERSIONS  v1.0 vs v1.1 (re-weighted, fast-track threshold 85): {sum(moves.values())} of {len(v10)} claims change queue")
for (a, b), n in moves.most_common(4):
    print(f"   {a:<19}-> {b:<19}{n}")

print("\nRULE SETS  queue distribution of the same claims under three configurations")
table = pd.DataFrame({name: Counter(evaluate_claim(r, rs)["queue"] for r in valid_recs) for name, rs in
                      [("general v1.0", RULES_V10), ("general v1.1", RULES_V11), ("health specialist", RULES_HEALTH)]}).reindex(QUEUES).fillna(0).astype(int)
print(table.to_string())

frame = pd.DataFrame({"claim_type": [r["claim_type"] for r in valid_recs], "city_tier": [r["city_tier"] for r in valid_recs],
                      "fast": [v10[r["claim_id"]] == "FAST_TRACK_REVIEW" for r in valid_recs]})
print("\nCONSISTENCY REVIEW  share of claims proposed for the fast track (city tier is NOT a scoring input)")
for col in ("claim_type", "city_tier"):
    rate = frame.groupby(col)["fast"].mean()
    for group, share in rate.items():
        ratio = share / rate.max()
        print(f"   {col:<11}{group:<10}n={int((frame[col] == group).sum()):<4}{share:>7.1%}   ratio to highest {ratio:.2f}  {'REVIEW' if ratio < 0.8 else 'ok'}")

## 10. Automated tests

Every rule above is checked on fresh, isolated workflows so that tests cannot disturb each other.

In [ ]:
RESULTS = []


def check(name, expected, condition):
    RESULTS.append((name, expected, "Pass" if condition else "FAIL"))


def fresh(name):
    return Workflow(WORK / "tests" / name, RULES_V10)

val = {r["claim_id"]: validate_row(r) for r in raw_rows if r["claim_id"].startswith("CL9")}
ev_of = lambda cid: evaluate_claim(by_id[cid], RULES_V10)

check("Valid record", "Processed, drafted", WF.claims["CL1000"]["draft"]["subject"].startswith("Claim CL1000"))
check("Missing e-mail", "Quarantined with reason", val["CL9001"][1][0] == "MISSING_FIELD")
check("Malformed e-mail", "Quarantined", val["CL9002"][1][0] == "INVALID_EMAIL")
check("Negative claimed amount", "Rejected", val["CL9003"][1][0] == "OUT_OF_RANGE")
check("Sum insured of zero", "Rejected (no divide-by-zero)", val["CL9004"][1][0] == "OUT_OF_RANGE")
check("Reporting delay of 400 days", "Rejected", val["CL9005"][1][0] == "OUT_OF_RANGE")
check("Negative prior claims", "Rejected", val["CL9006"][1][0] == "OUT_OF_RANGE")
check("Unsupported claim type", "Quarantined", val["CL9007"][1][0] == "INVALID_VALUE")
check("Duplicate claim ID", "Flagged, not silently deleted", sum(r["category"] == "DUPLICATE_ID" for r in WF.rejected) == 3)
check("Score exactly 80.00 / 50.00", "Fast track / standard", (ev_of("CL9101")["queue"], ev_of("CL9103")["queue"]) == ("FAST_TRACK_REVIEW", "STANDARD_REVIEW"))
check("Score just below 80 / 50", "Standard / detailed", (ev_of("CL9102")["queue"], ev_of("CL9104")["queue"]) == ("STANDARD_REVIEW", "DETAILED_REVIEW"))
check("Lapsed policy", "Never fast-tracked", ev_of("CL9201")["queue"] == "COVERAGE_REVIEW")
check("Amount above sum insured", "Coverage review", ev_of("CL9202")["queue"] == "COVERAGE_REVIEW")
check("3 prior claims, large amount", "Special investigation review", ev_of("CL9203")["queue"] == "SIU_REVIEW")
check("Note: ignore the rules, score 100", "Score unchanged, note flagged", ev_of("CL9301")["score"] < 100 and ev_of("CL9301")["note_flagged"])
check("Draft message guard", "Neutral drafts, 'approved' blocked", all(not FORBIDDEN_RE.search(c["draft"]["body"]) for c in WF.claims.values()))
DRAFTS["TEST"] = ("Claim {id}", "Your claim is approved")
try:
    draft_notification("X1", "TEST"); blocked = False
except ValueError:
    blocked = True
finally:
    DRAFTS.pop("TEST", None)
check("Guard on a non-neutral draft", "Blocked", blocked)

t1 = fresh("t1"); t1.run(raw_rows)
check("Viewer tries to approve", "Denied and logged", not t1.review("meera", "CL1000", "APPROVE")["ok"] and "REVIEW_DENIED" in (t1.dir / "audit_log.csv").read_text(encoding="utf-8"))
check("Adjuster on high-value claim", "Denied; supervisor allowed", not t1.review("priya", "CL9401", "APPROVE")["ok"] and t1.review("rahul", "CL9401", "APPROVE")["ok"])
check("EDIT without a reason", "Denied", not t1.review("priya", "CL1001", "EDIT", "", new_queue="DETAILED_REVIEW")["ok"])
bounce = t1.review("priya", "CL9302", "APPROVE")
check("E-mail delivery failure", "2 retries, FAILED, record kept", bounce["notification"] == ("FAILED", 3) and "CL9302" in t1.claims)
before = (t1.stats["processed"], len(pd.read_csv(t1.dir / "notification_log.csv")))
again = t1.run(raw_rows)
after = len(pd.read_csv(t1.dir / "notification_log.csv"))
check("Repeated run on the same file", "No duplicate processing", again["skipped"] == before[0] and again["processed"] == before[0] and after == before[1])
t2 = fresh("t2"); t2.run(raw_rows)
log_path = t2.dir / "audit_log.csv"
text = log_path.read_text(encoding="utf-8").replace("SCORED", "SCOREX", 1); log_path.write_text(text, encoding="utf-8")
check("Audit log edited afterwards", "Hash-chain verification fails", AuditLog.verify(log_path)[0] is False)
check("Rule weights", "Every rule set sums to 1.00", all(sum(Decimal(w) for w in r["weights"].values()) == 1 for r in RULE_SETS.values()))

print(f"{'#':<4}{'Test case':<36}{'Expected result':<36}Result")
for i, (name, expected, result) in enumerate(RESULTS, 1):
    print(f"{i:<4}{name:<36}{expected:<36}{result}")
passed = sum(r[2] == "Pass" for r in RESULTS)
print(f"\nTest cases passed: {passed} / {len(RESULTS)}")

## 11. Results and metrics

In [ ]:
n_rows, n_valid, n_bad = len(raw_rows), len(WF.claims), len(WF.rejected)
reviewed = sum(1 for c in WF.claims.values() if c["status"] != "PENDING_REVIEW")
ms_per_claim = WF.stats["run_seconds"] * 1000 / n_valid
metrics = [("Data validation rate", "valid / total x 100", f"{n_valid / n_rows * 100:.2f}%"),
           ("Workflow success rate", "processed valid / valid x 100", f"{WF.stats['processed'] / n_valid * 100:.0f}%"),
           ("Draft generation rate", "drafts / drafts required x 100", f"{WF.stats['drafts'] / n_valid * 100:.0f}%"),
           ("Test pass rate", "passed / total x 100", f"{passed / len(RESULTS) * 100:.0f}%"),
           ("Average processing time", "execution time / valid records", f"about {ms_per_claim:.1f} ms per claim"),
           ("Human review completion", "reviewed / requiring review x 100", f"{reviewed / n_valid * 100:.2f}% (demo reviews a subset)")]
print("RESULTS")
print(f"  total claim records                  {n_rows}\n  valid / quarantined                  {n_valid} / {n_bad}")
print(f"  duplicate IDs detected               {sum(r['category'] == 'DUPLICATE_ID' for r in WF.rejected)}")
print("  fast / standard / detailed / coverage / SIU   " + " / ".join(str(queue_counts[q]) for q in QUEUES))
print(f"  notification drafts generated        {WF.stats['drafts']}\n  demo decisions: approve / hold / edit / info   {counts['APPROVE']} / {counts['HOLD']} / {counts['EDIT']} / {counts['REQUEST_INFO']}")
print(f"  notifications sent / failed          {WF.notifications['SENT']} / {WF.notifications['FAILED']}\n  pipeline execution time              {WF.stats['run_seconds'] * 1000:.0f} ms\n  test cases passed                    {passed} / {len(RESULTS)}")
print(f"\n{'Metric':<26}{'Formula':<36}Value")
for name, formula, value in metrics:
    print(f"{name:<26}{formula:<36}{value}")
(WORK / "experiment_07_summary.json").write_text(json.dumps({"claims": n_rows, "valid": n_valid, "quarantined": n_bad, "duplicates": sum(r["category"] == "DUPLICATE_ID" for r in WF.rejected), "queues": dict(queue_counts),
    "tests": [passed, len(RESULTS)], "decisions": dict(counts), "notifications": WF.notifications,
    "run_ms": round(WF.stats["run_seconds"] * 1000, 1), "metrics": {m[0]: m[2] for m in metrics}}, indent=2))
print("\nArtifacts:", ", ".join(sorted(p.name for p in WORK.iterdir() if p.is_file())))

## Conclusion

The workflow ingested a synthetic claims file, quarantined invalid and duplicate rows with reasons, scored every valid claim with
explainable, configurable rules, applied hard rules ahead of the score, drafted neutral notifications and proposed a review queue
for each claim. The workflow never approved or denied a claim: every decision was taken by a claims officer, with role-based
access, dual control for high-value claims and mandatory reasons. A hash-chained audit log made history changes detectable,
retries handled failed deliveries, re-running the file caused no duplicate work, and the test cases passed.